# Imports

In [4]:
import pandas as pd
from pathlib import Path

root = Path("dataset/raw")
df = pd.read_csv(root / "data.csv")

for c in ["ref_path", "left_path", "right_path"]:
    df[c] = df[c].str.replace("util/2afc_src_images/", "", regex=False)
df = df[df["left_vote"] != df["right_vote"]].copy()
df["choice"] = (df["right_vote"] > df["left_vote"]).astype(int)

triplets = pd.DataFrame({
    "reference": df["ref_path"],
    "candidate_a": df["left_path"],
    "candidate_b": df["right_path"],
    "choice": df["choice"]
})

exists = triplets.apply(
lambda r: (root / r["reference"]).exists()
and (root / r["candidate_a"]).exists()
and (root / r["candidate_b"]).exists(),
axis=1
)
triplets = triplets[exists].reset_index(drop=True)

print("usable triplets:", len(triplets))
print(triplets["choice"].value_counts(dropna=False))

triplets.to_csv("dataset/processed/triplets_available.csv", index=False)

usable triplets: 5364
choice
1    2785
0    2579
Name: count, dtype: int64


In [ ]:
# curl -fL --retry 5 --retry-delay 2 -o data.csv https://data.csail.mit.edu/nights/nights_unfiltered/nights_raw.csv
# python ../src/datasets/split_builder.py --triplets_csv dataset/processed/triplets_available.csv --out_dir splits --seed 42

In [1]:
from src.datasets.nights_triplet_dataset import NightsTripletDataset
from src.datasets.transforms import build_transforms

ds = NightsTripletDataset(
    split_csv="dataset/splits/train.csv",
    images_root="dataset/raw",
    transform=build_transforms(image_size=224, train=True),
)

print("dataset size:", len(ds))
sample = ds[0]
print("reference:", tuple(sample["reference"].shape))
print("candidate_a:", tuple(sample["candidate_a"].shape))
print("candidate_b:", tuple(sample["candidate_b"].shape))
print("choice:", sample["choice"])
print("paths:", sample["reference_path"], sample["candidate_a_path"], sample["candidate_b_path"])

dataset size: 3754
reference: (3, 224, 224)
candidate_a: (3, 224, 224)
candidate_b: (3, 224, 224)
choice: 0
paths: dataset\raw\ref\013\534.png dataset\raw\distort\013\534_0.png dataset\raw\distort\013\534_1.png


# Globals

# Utils

# Data

# Network

# Train

# Test